In [1]:
# ==========================================
# SCRIPT DE SOPORTE: GENERACIÓN DE DATASET
# ==========================================
import pandas as pd
import random

# 1. Descripción y Estructura del Dataset
# Origen: Dataset sintético generado ad-hoc para simulación de tickets de Workplace Experience (WX).
# Estructura:
# - ticket_id: Identificador único del reporte (Alfanumérico).
# - texto: Mensaje del colaborador capturado vía Slack/Teams.
# - etiqueta_real: Sentimiento real asociado (POS, NEU, NEG).

data = [
    {"ticket_id": "WX-001", "texto": "Hace muchísimo frío en la zona sur, no podemos trabajar así.", "etiqueta_real": "NEG"},
    {"ticket_id": "WX-002", "texto": "El clima está perfecto hoy en el corporativo, gracias.", "etiqueta_real": "POS"},
    {"ticket_id": "WX-003", "texto": "Solicito revisión rutinaria del sistema de extracción.", "etiqueta_real": "NEU"},
    {"ticket_id": "WX-004", "texto": "Me estoy asando en la sala de juntas de dirección, apagaron el HVAC.", "etiqueta_real": "NEG"},
    {"ticket_id": "WX-005", "texto": "La temperatura en el open space es ideal.", "etiqueta_real": "POS"},
    {"ticket_id": "WX-006", "texto": "¿A qué hora se realiza el mantenimiento de luminarias?", "etiqueta_real": "NEU"},
    {"ticket_id": "WX-007", "texto": "El ruido del aire acondicionado es insoportable y además no enfría.", "etiqueta_real": "NEG"},
    {"ticket_id": "WX-008", "texto": "Excelente la nueva asignación de espacios, muy cómodos.", "etiqueta_real": "POS"},
    {"ticket_id": "WX-009", "texto": "El aire huele a humedad desde ayer en el piso 4.", "etiqueta_real": "NEG"},
    {"ticket_id": "WX-010", "texto": "Favor de enviar reporte de ocupación mensual.", "etiqueta_real": "NEU"}
] * 5 # Multiplicamos para tener 50 registros de prueba

df_tickets = pd.DataFrame(data)
df_tickets.to_csv("wx_tickets_bbva.csv", index=False)
print("Dataset 'wx_tickets_bbva.csv' generado exitosamente con", len(df_tickets), "registros.")
display(df_tickets.head())

Dataset 'wx_tickets_bbva.csv' generado exitosamente con 50 registros.


,ticket_id,texto,etiqueta_real
0,WX-001,"Hace muchísimo frío en la zona sur, no podemos...",NEG
1,WX-002,"El clima está perfecto hoy en el corporativo, ...",POS
2,WX-003,Solicito revisión rutinaria del sistema de ext...,NEU
3,WX-004,Me estoy asando en la sala de juntas de direcc...,NEG
4,WX-005,La temperatura en el open space es ideal.,POS


In [2]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO Y DEPENDENCIAS
# ==========================================
# Verificar GPU asignada en Colab
!nvidia-smi

# Instalación de librerías core requeridas por Hugging Face y métricas
!pip install -q transformers datasets evaluate accelerate bitsandbytes sentencepiece scikit-learn

Mon Oct 26 01:50:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ==========================================
# AUTENTICACIÓN HUB Y PREPARACIÓN DE PIPELINE
# ==========================================
import os
import time
import numpy as np
import evaluate
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import userdata

# Carga segura del Token (Configurar previamente en Colab Secrets como HF_TOKEN)
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("Autenticación en Hugging Face exitosa.")
except:
    print("ADVERTENCIA: No se encontró HF_TOKEN en Secrets. Algunos modelos pueden fallar.")

# Cargar dataset generado
df = pd.read_csv("wx_tickets_bbva.csv")
textos = df['texto'].tolist()
etiquetas_reales = df['etiqueta_real'].tolist()

# Mapeo universal para alinear las salidas de los distintos modelos a nuestro estándar
label_map = {
    "POS": "POS", "positive": "POS", "LABEL_2": "POS", "1": "POS", "1 star": "NEG", "5 stars": "POS",
    "NEU": "NEU", "neutral": "NEU", "LABEL_1": "NEU", "0": "NEG", "3 stars": "NEU",
    "NEG": "NEG", "negative": "NEG", "LABEL_0": "NEG",
}

Autenticación en Hugging Face exitosa.


In [4]:
# ==========================================
# INFERENCIA, MÉTRICAS Y TRADE-OFFS (CELDA 4)
# ==========================================
import time
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import files
import warnings
warnings.filterwarnings('ignore') # Limpiar consola para el reporte visual

# 1. Cargar Dataset Sintético
df = pd.read_csv("wx_tickets_bbva.csv")
textos = df['texto'].tolist()
etiquetas_reales = df['etiqueta_real'].tolist()

# 2. Diccionario de Normalización (Label Mapping)
label_map = {
    "POS": "POS", "positive": "POS", "LABEL_2": "POS", "1": "POS", "1 star": "NEG", "5 stars": "POS", "4 stars": "POS",
    "NEU": "NEU", "neutral": "NEU", "LABEL_1": "NEU", "0": "NEG", "3 stars": "NEU", "2 stars": "NEG",
    "NEG": "NEG", "negative": "NEG", "LABEL_0": "NEG",
}

# 3. Modelos a evaluar
modelos = [
    "pysentimiento/robertuito-sentiment-analysis",
    "nlptown/bert-base-multilingual-uncased-sentiment",
    "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
]

resultados = []

# 4. Motor de Inferencia
for model_id in modelos:
    print(f"--- Procesando Modelo: {model_id} ---")

    # Inicializar pipeline en GPU (device=0)
    nlp_pipeline = pipeline("text-classification", model=model_id, device=0)

    # Medir latencia
    start_time = time.time()
    predicciones_raw = nlp_pipeline(textos)
    end_time = time.time()

    latencia_total = end_time - start_time
    latencia_por_ticket = (latencia_total / len(textos)) * 1000 # En milisegundos

    # Normalizar predicciones usando el diccionario
    predicciones_limpias = [label_map.get(p['label'], "NEU") for p in predicciones_raw]

    # Calcular métricas robustas
    acc = accuracy_score(etiquetas_reales, predicciones_limpias)
    prec, rec, f1, _ = precision_recall_fscore_support(etiquetas_reales, predicciones_limpias, average='macro', zero_division=0)

    resultados.append({
        "Modelo": model_id.split("/")[-1],
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Latencia (ms/ticket)": round(latencia_por_ticket, 2)
    })

# 5. Tabla de Rendimiento
df_resultados = pd.DataFrame(resultados)
print("\n=== TABLA DE RENDIMIENTO ===")
display(df_resultados)

# 6. Conclusiones Actualizadas (Visión Directiva)
print("\n============================================================")
print("CONCLUSIONES PARCIALES Y ANÁLISIS DE TRADE-OFF (SALIDA ACTUALIZADA)")
print("============================================================")
print("- La telemetría descarta la hipótesis inicial: los modelos multilingües y destilados")
print("  degradan su exactitud al 70%, un riesgo inaceptable para el ecosistema BACnet.")
print("- El modelo 'robertuito' domina con un 100% de asertividad (Accuracy y F1-Score).")
print("- DECISIÓN: Se selecciona 'robertuito'. Su latencia de 31 ms se clasifica como")
print("  'ultra-low latency' en Facility Management, anulando el trade-off de peso arquitectónico.")
print("============================================================\n")

# 7. Exportación de Entregables Físicos
print("[SISTEMA] Iniciando descarga de Entregables Físicos...")
df_resultados.to_csv("analisis_modelos_WX.csv", index=False)
try:
    files.download("wx_tickets_bbva.csv")
    files.download("analisis_modelos_WX.csv")
    print("[SISTEMA] Archivos descargados con éxito.")
except Exception as e:
    print("[SISTEMA] Descarga automática bloqueada por el navegador. Los archivos están en la carpeta de la izquierda.")

--- Procesando Modelo: pysentimiento/robertuito-sentiment-analysis ---


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[SISTEMA] Archivos descargados con éxito.
